# Cleaning and Combining

Trap counts (the A2 data) and the weather at The Forks. Each step: what, why, cost, check.

In [1]:
import pandas as pd

from pathlib import Path

# Settings: change these, not the code below
TRAP_FILE = "../earlier-work/a1-mosquitoes/data/raw/traps-2015-2025.csv"
WEATHER_FOLDER = "../data/raw/weather"
SPRING_MONTHS = [5, 6]          # May and June: the rain that fills the ponds
DAILY_FILE = "../data/clean/daily.csv"
SEASONS_FILE = "../data/clean/seasons.csv"

In [2]:
traps_raw = pd.read_csv(TRAP_FILE)
weather_raw = pd.concat([pd.read_csv(f) for f in sorted(Path(WEATHER_FOLDER).glob("*.csv"))], ignore_index=True)
print(len(traps_raw), "trap rows,", len(weather_raw), "weather rows")

880 trap rows, 4018 weather rows


**Step 1: dates, day first.** What: parse the trap dates as day first. Why: the file writes `13/05/2015`; read
month first, it fails or swaps days and months. Cost: none. Check: every count date is May to October.

In [3]:
traps = traps_raw.rename(columns={"City Wide Daily Average": "city_avg", "North West Average": "north_west",
                                  "North East Average": "north_east", "South East Average": "south_east",
                                  "South West Average": "south_west"})
traps["date"] = pd.to_datetime(traps["Count Date"], dayfirst=True)
traps = traps[["date", "city_avg", "north_west", "north_east", "south_east", "south_west"]]
assert traps["date"].dt.month.between(5, 10).all()
assert traps.duplicated(subset=["date"]).sum() == 0
traps["date"].dt.year.value_counts().sort_index()

date
2015    102
2016     86
2017     61
2018     61
2019     77
2020     76
2021     60
2022    118
2023     73
2024    107
2025     59
Name: count, dtype: int64

**Step 2: the weather, three columns.** What: keep the date, mean temperature and total precipitation. Why: that's
all the story uses. Cost: the other columns (snow, degree days) aren't checked. Check: one row per day.

In [4]:
weather = pd.DataFrame({
    "date": pd.to_datetime(weather_raw["Date/Time"]),
    "mean_temp": weather_raw["Mean Temp (°C)"],
    "precip_mm": weather_raw["Total Precip (mm)"],
})
assert weather.duplicated(subset=["date"]).sum() == 0
print(weather["date"].min().date(), "to", weather["date"].max().date())

2015-01-01 to 2025-12-31


**Step 3: join each count day to that day's weather.** Every count day should find one.

In [5]:
daily = traps.merge(weather, on="date", how="left", indicator="source")
missing = sorted(set(traps["date"]) - set(weather["date"]))
assert missing == [], missing
daily["source"].value_counts()

source
both          880
left_only       0
right_only      0
Name: count, dtype: int64